# 第十周：当前队列鼠级 PCA 衰老方向

按用户确认，主分析保留08原117个鼠×节点、22个mouse.id统计单元，PC1–7；30/50细胞只作敏感性。young/old各≥2个mouse.id的7节点进入正式热图，Liver lymphoid探索性。24m=middle、30m=old沿用06；所有Python逻辑内嵌，读写用相对地址字面量。


In [1]:
import json
import hashlib
import importlib.metadata
import warnings
import gc
import h5py
import anndata as ad
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy.stats import spearmanr, trim_mean
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import squareform
from threadpoolctl import threadpool_limits
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["font.sans-serif"]=["Microsoft YaHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
plt.rcParams["pdf.fonttype"]=42
SEED=2026
NODES=["Kidney | lymphoid","Kidney | myeloid","Liver | lymphoid","Liver | myeloid","Lung | lymphoid","Lung | myeloid","spleen/marrow | lymphoid","spleen/marrow | myeloid"]

def axis_cos(a,b):
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den>1e-12 else np.nan

def fit_axis(frame,k=7,mode="centroid",weights=None):
    cols=[f"PC{i+1}" for i in range(k)]
    s=frame.copy()
    w=np.ones(len(s)) if weights is None else s.mouse_id.map(weights).fillna(0).to_numpy(float)
    keep=w>0;s=s.loc[keep].copy();w=w[keep]
    sizes=s.age_group.value_counts()
    ny,no=int(sizes.get("young",0)),int(sizes.get("old",0))
    result={"n_young":ny,"n_old":no,"n_middle":int(sizes.get("middle",0)),"status":"not_estimable","reason":"","formal":ny>=2 and no>=2}
    if not ny or not no:
        result["reason"]="missing_young_or_old";return result
    y=s.age_group.eq("young").to_numpy();o=s.age_group.eq("old").to_numpy()
    X=s[cols].to_numpy(float)
    mu_y=np.average(X[y],axis=0,weights=w[y])
    d=np.average(X[o],axis=0,weights=w[o])-mu_y
    if mode=="sex_adjusted":
        terms=[np.ones(len(s))]
        if s.sex.nunique()>1:terms.append(s.sex.eq("female").to_numpy(float))
        if s.age_group.eq("middle").any():terms.append(s.age_group.eq("middle").to_numpy(float))
        terms.append(o.astype(float))
        design=np.column_stack(terms);rootw=np.sqrt(w)
        rank=np.linalg.matrix_rank(design*rootw[:,None])
        result.update(design_rank=int(rank),design_columns=design.shape[1],residual_df=int(len(s)-rank))
        if rank<design.shape[1] or len(s)<=rank:
            result["reason"]="rank_deficient_or_no_residual_df";return result
        coef=np.linalg.lstsq(design*rootw[:,None],X*rootw[:,None],rcond=None)[0]
        d=coef[-1];mu_y=coef[0]
        result["sex_coefficient"]=coef[1] if s.sex.nunique()>1 else np.zeros(k)
    norm=float(np.linalg.norm(d))
    if not np.isfinite(norm) or norm<=1e-12:
        result["reason"]="zero_or_nonfinite_direction";return result
    result.update(d=d,mu_y=mu_y,norm=norm,status="complete" if result["formal"] else "exploratory_low_replication")
    return result

def project_axis(frame,fit,k):
    cols=[f"PC{i+1}" for i in range(k)]
    if "d" not in fit:return np.full(len(frame),np.nan)
    X=frame[cols].to_numpy(float)
    # Standardise held-out PCs to male reference using training-only sex coefficients.
    if "sex_coefficient" in fit:X=X-frame.sex.eq("female").to_numpy(float)[:,None]*fit["sex_coefficient"]
    return (X-fit["mu_y"])@fit["d"]/fit["norm"]

def describe_age(scores,ages):
    s=np.asarray(scores,float);a=np.asarray(ages,float);valid=np.isfinite(s)&np.isfinite(a)
    return float(spearmanr(s[valid],a[valid]).statistic) if valid.sum()>=4 and np.unique(s[valid]).size>1 and np.unique(a[valid]).size>1 else np.nan

def aggregate_coordinates(obs,coords,aggregation="mean",cell_mask=None,cap=None,seed=2026):
    keep=np.ones(len(obs),bool) if cell_mask is None else np.asarray(cell_mask,bool)
    sub=obs.loc[keep].copy().reset_index(drop=True)
    z=np.asarray(coords)[keep]
    groups=sub.groupby(["mouse_id","tissue","mct"],observed=True,sort=True).indices
    rng=np.random.default_rng(seed);rows=[]
    for key,idx in groups.items():
        if cap is not None and len(idx)>cap:idx=rng.choice(idx,size=cap,replace=False)
        block=z[idx]
        vector=block.mean(0) if aggregation=="mean" else trim_mean(block,.1,axis=0)
        meta=sub.iloc[idx[0]]
        rows.append(dict(mouse_id=str(key[0]),tissue=str(key[1]),mct=str(key[2]),node=str(key[1])+" | "+str(key[2]),age=int(meta.age),age_group=str(meta.age_group),sex=str(meta.sex),n_cells=len(idx),**{f"PC{i+1}":float(v) for i,v in enumerate(vector)}))
    return pd.DataFrame(rows)

def cohort_filter(table,keys,min_cells):
    valid=table.apply(lambda r:(r.mouse_id,r.tissue,r.mct) in keys,axis=1)
    return table.loc[valid & table.n_cells.ge(min_cells)].copy()

def evaluate_axes(table,k=7,mode="centroid"):
    fits={};summ=[];fullscores=[];lomo=[]
    for node in NODES:
        g=table.loc[table.node.eq(node)].copy()
        fit=fit_axis(g,k,mode);fits[node]=fit
        row={key:value for key,value in fit.items() if key not in ["d","mu_y","sex_coefficient"]}
        row.update(node=node,n_mice=len(g),n_cells=int(g.n_cells.sum()) if len(g) else 0,median_cells=float(g.n_cells.median()) if len(g) else np.nan)
        if "d" in fit:
            gs=g.copy();gs["in_sample_score"]=project_axis(g,fit,k);fullscores.append(gs)
            for mouse in g.mouse_id:
                train=g.loc[g.mouse_id.ne(mouse)]
                held=g.loc[g.mouse_id.eq(mouse)];lf=fit_axis(train,k,mode)
                item=dict(node=node,mouse_id=mouse,age=int(held.age.iloc[0]),age_group=held.age_group.iloc[0],sex=held.sex.iloc[0],status=lf["status"],reason=lf["reason"],n_train_young=lf["n_young"],n_train_old=lf["n_old"],score=float(project_axis(held,lf,k)[0]),cosine_to_full=axis_cos(fit["d"],lf["d"]) if "d" in lf else np.nan)
                lomo.append(item)
            items=[x for x in lomo if x["node"]==node]
            row["rho_age_crossfit"]=describe_age([r["score"] for r in items],[r["age"] for r in items])
            vals=[r["cosine_to_full"] for r in items if np.isfinite(r["cosine_to_full"])]
            row["min_lomo_cosine"]=min(vals) if vals else np.nan
            row["median_lomo_cosine"]=float(np.median(vals)) if vals else np.nan
            row["n_lomo_estimable"]=len(vals)
        summ.append(row)
    eligible=[n for n in NODES if fits[n]["formal"] and "d" in fits[n]]
    cosine=pd.DataFrame([[axis_cos(fits[a]["d"],fits[b]["d"]) for b in eligible] for a in eligible],index=eligible,columns=eligible)
    return {"fits":fits,"summary":pd.DataFrame(summ),"scores":pd.concat(fullscores,ignore_index=True) if fullscores else pd.DataFrame(),"lomo":pd.DataFrame(lomo),"cosine":cosine,"eligible":eligible}

def compare_scenarios(reference,trial):
    common=[n for n in reference["eligible"] if n in trial["eligible"]]
    corr=np.nan;sign=np.nan;maxdiff=np.nan
    if len(common)>=3:
        idx=np.triu_indices(len(common),1)
        a=reference["cosine"].loc[common,common].to_numpy()[idx]
        b=trial["cosine"].loc[common,common].to_numpy()[idx]
        corr=describe_age(b,a)
        sign=float(np.mean(np.sign(a)==np.sign(b)));maxdiff=float(np.max(abs(a-b)))
    return dict(n_common_nodes=len(common),cosine_matrix_spearman=corr,pair_sign_agreement=sign,max_pair_cosine_change=maxdiff)

def full_bootstrap(table,k=7,B=500,seed=2026):
    base=evaluate_axes(table,k);mice=table[["mouse_id","age_group","sex"]].drop_duplicates("mouse_id")
    rng=np.random.default_rng(seed);drawrows=[];pairrows=[]
    groups={n:table.loc[table.node.eq(n)] for n in NODES}
    for b in range(B):
        draws=[]
        for age in ["young","middle","old"]:
            pool=mice.loc[mice.age_group.eq(age),"mouse_id"].to_numpy()
            if len(pool):draws.extend(rng.choice(pool,len(pool),replace=True))
        weights=pd.Series(draws).value_counts().to_dict();fits={}
        for n in base["eligible"]:
            fit=fit_axis(groups[n],k,weights=weights);fits[n]=fit
            drawrows.append(dict(iteration=b,node=n,status=fit["status"],reason=fit["reason"],n_unique_young=fit["n_young"],n_unique_old=fit["n_old"],direction_norm=fit.get("norm",np.nan),cosine_to_baseline=axis_cos(base["fits"][n]["d"],fit["d"]) if "d" in fit else np.nan))
        for i,a in enumerate(base["eligible"]):
            for c in base["eligible"][i+1:]:
                value=axis_cos(fits[a]["d"],fits[c]["d"]) if "d" in fits[a] and "d" in fits[c] else np.nan
                pairrows.append(dict(iteration=b,node1=a,node2=c,cosine=value))
    return pd.DataFrame(drawrows),pd.DataFrame(pairrows)


In [2]:
# Frozen 06 metadata and PCA; no age remapping or old Harmony inputs.
! powershell.exe -NoProfile -Command "New-Item -ItemType Directory -Force '../results/aging_axis/current_cohort_20261030', '../results/pca', '../figures/main' | Out-Null"
with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    frozen_obs=read_elem(handle["obs"])
    frozen_coords=np.asarray(read_elem(handle["obsm"])["X_pca"])
    frozen_var=read_elem(handle["var"])
    frozen_pca_variance=read_elem(handle["uns/pca/variance_ratio"])
with h5py.File("../data_processed/pseudobulk.h5ad","r") as handle:
    pb_obs=read_elem(handle["obs"])
frozen_obs=frozen_obs.rename(columns={"mouse.id":"mouse_id"}).copy()
for key in ["mouse_id","tissue","mct","age_group","sex"]:frozen_obs[key]=frozen_obs[key].astype(str)
frozen_obs["age"]=pd.to_numeric(frozen_obs.age)
assert len(frozen_obs)==96569 and frozen_coords.shape==(96569,50)
assert np.isfinite(frozen_coords).all()
assert not (frozen_obs.groupby("mouse_id")[["age","age_group","sex"]].nunique()>1).any().any()
assert frozen_obs.loc[frozen_obs.age.eq(24),"age_group"].eq("middle").all()
assert frozen_obs.loc[frozen_obs.age.eq(30),"age_group"].eq("old").all()
cohort_meta=pb_obs[["mouse.id","tissue","mct","age","age_group","sex"]].drop_duplicates().rename(columns={"mouse.id":"mouse_id"})
cohort_meta["mouse_id"]=cohort_meta.mouse_id.astype(str)
cohort_keys=set(cohort_meta[["mouse_id","tissue","mct"]].itertuples(index=False,name=None))
assert len(cohort_keys)==117
mouse_all=aggregate_coordinates(frozen_obs,frozen_coords)
mouse_mean=cohort_filter(mouse_all,cohort_keys,0)
mouse_trim=cohort_filter(aggregate_coordinates(frozen_obs,frozen_coords,"trimmed"),cohort_keys,0)
assert len(mouse_mean)==117 and mouse_mean.mouse_id.nunique()==22
mouse_all.to_csv("../results/aging_axis/current_cohort_20261030/all_mouse_node_PCA.tsv",sep="\t",index=False)
mouse_mean.to_csv("../results/aging_axis/current_cohort_20261030/mouse_PCA_mean.tsv",sep="\t",index=False)
mouse_trim.to_csv("../results/aging_axis/current_cohort_20261030/mouse_PCA_trimmed.tsv",sep="\t",index=False)
mouse_mean.to_csv("../results/pca/mouse_level_pca.csv",index=False)
mouse_all.loc[~mouse_all.apply(lambda r:(r.mouse_id,r.tissue,r.mct) in cohort_keys,axis=1)].to_csv("../results/aging_axis/current_cohort_20261030/outside_current_DE_cohort.tsv",sep="\t",index=False)
coverage=[]
for threshold in [0,30,50]:
    for node in NODES:
        g=mouse_mean.loc[mouse_mean.node.eq(node)&mouse_mean.n_cells.ge(threshold)]
        count=g.age_group.value_counts()
        coverage.append(dict(node=node,min_cells=threshold,n_samples=len(g),n_young=int(count.get("young",0)),n_middle=int(count.get("middle",0)),n_old=int(count.get("old",0)),formal=int(count.get("young",0))>=2 and int(count.get("old",0))>=2))
pd.DataFrame(coverage).to_csv("../results/aging_axis/current_cohort_20261030/node_coverage_by_min_cells.tsv",sep="\t",index=False)
print(pd.DataFrame(coverage).to_string(index=False))


                    node  min_cells  n_samples  n_young  n_middle  n_old  formal
       Kidney | lymphoid          0         16        5         7      4    True
        Kidney | myeloid          0         16        5         7      4    True
        Liver | lymphoid          0          7        1         3      3   False
         Liver | myeloid          0          8        2         3      3    True
         Lung | lymphoid          0         16        6         6      4    True
          Lung | myeloid          0         16        6         6      4    True
spleen/marrow | lymphoid          0         19        5        10      4    True
 spleen/marrow | myeloid          0         19        5        10      4    True
       Kidney | lymphoid         30         10        0         6      4   False
        Kidney | myeloid         30         15        4         7      4    True
        Liver | lymphoid         30          5        1         2      2   False
         Liver | myeloid    

In [3]:
# User-confirmed primary includes the complete 08 mouse-node cohort; 30/50 cells are sensitivity settings.
MAIN_MIN_CELLS=0
MAIN_N_PC=7
main_samples=mouse_mean.loc[mouse_mean.n_cells.ge(MAIN_MIN_CELLS)].copy()
main_axis=evaluate_axes(main_samples,MAIN_N_PC)
main_summary=main_axis["summary"]
main_summary["sampling_unit"]="mouse.id identifier; composite IDs audited separately"
main_summary["interpretation"]=np.where(main_summary.formal,"eligible_for_primary_similarity","excluded_from_primary_similarity")
main_summary.to_csv("../results/aging_axis/current_cohort_20261030/node_summary.tsv",sep="\t",index=False)
main_summary.to_csv("../results/aging_axis/node_summary.tsv",sep="\t",index=False)
score_table=main_samples.merge(main_axis["scores"][["node","mouse_id","in_sample_score"]],on=["node","mouse_id"],how="left",validate="one_to_one")
score_table=score_table.merge(main_axis["lomo"][["node","mouse_id","score","status","reason","cosine_to_full","n_train_young","n_train_old"]],on=["node","mouse_id"],how="left",validate="one_to_one").rename(columns={"score":"held_out_score","status":"held_out_status"})
score_table["held_out_status"]=score_table.held_out_status.fillna("not_estimable")
score_table.to_csv("../results/aging_axis/current_cohort_20261030/pca_scores.csv",index=False)
score_table.to_csv("../results/aging_axis/pca_scores.csv",index=False)
main_axis["lomo"].to_csv("../results/aging_axis/current_cohort_20261030/leave_one_mouse_out.tsv",sep="\t",index=False)
main_axis["cosine"].to_csv("../results/aging_axis/current_cohort_20261030/cosine_similarity.csv")
main_axis["cosine"].to_csv("../results/aging_axis/cosine_similarity.csv")
direction_rows=[]
for node,fit in main_axis["fits"].items():
    direction_rows.append(dict(node=node,status=fit["status"],formal=fit["formal"],**{f"PC{i+1}":fit["d"][i] if "d" in fit else np.nan for i in range(MAIN_N_PC)}))
pd.DataFrame(direction_rows).to_csv("../results/aging_axis/current_cohort_20261030/aging_directions.csv",index=False)
# Sex-adjusted and male-only are controls, not replacements for the task-book centroid definition.
sex_control=evaluate_axes(main_samples,MAIN_N_PC,"sex_adjusted")
male_control=evaluate_axes(main_samples.loc[main_samples.sex.eq("male")],MAIN_N_PC)
control_rows=[]
for label,result in [("sex_adjusted",sex_control),("male_only",male_control)]:
    for node in NODES:
        a,b=main_axis["fits"][node],result["fits"][node]
        control_rows.append(dict(control=label,node=node,status=b["status"],n_young=b["n_young"],n_old=b["n_old"],formal=b["formal"],cosine_to_main=axis_cos(a["d"],b["d"]) if "d" in a and "d" in b else np.nan,rho_age_crossfit=result["summary"].set_index("node").loc[node].get("rho_age_crossfit",np.nan)))
pd.DataFrame(control_rows).to_csv("../results/aging_axis/current_cohort_20261030/sex_sensitivity.tsv",sep="\t",index=False)
# Independent small analytical examples required by week 10.
toy=pd.DataFrame({"mouse_id":["y1","y2","o1","o2","m1","m2"],"age_group":["young","young","old","old","middle","middle"],"sex":["male"]*6,"PC1":[-1,1,-1,1,0,0],"PC2":[0,0,2,2,1,1]})
toy_fit=fit_axis(toy,2)
tests=[]
def record_test(name,passed):tests.append(dict(check=name,passed=bool(passed)))
record_test("analytical_old_minus_young",np.allclose(toy_fit["d"],[0,2]))
record_test("young_origin",np.allclose(project_axis(toy.iloc[:2],toy_fit,2),0))
record_test("old_score_equals_length",np.allclose(project_axis(toy.iloc[2:4],toy_fit,2),toy_fit["norm"]))
alter=toy.copy();alter.loc[alter.age_group.eq("middle"),["PC1","PC2"]]=10000
record_test("middle_not_used_in_centroid_direction",np.allclose(fit_axis(alter,2)["d"],toy_fit["d"]))
before=fit_axis(toy.loc[toy.mouse_id.ne("y1")],2)
alter=toy.copy();alter.loc[alter.mouse_id.eq("y1"),"PC1"]=9999
after=fit_axis(alter.loc[alter.mouse_id.ne("y1")],2)
record_test("held_mouse_not_used_in_training_centres",np.allclose(before["d"],after["d"]) and np.allclose(before["mu_y"],after["mu_y"]))
record_test("symmetry_and_diagonal",np.allclose(main_axis["cosine"],main_axis["cosine"].T) and np.allclose(np.diag(main_axis["cosine"]),1))
record_test("one_row_per_mouse_node",not main_samples.duplicated(["node","mouse_id"]).any())
record_test("same_mouse_metadata_across_tissues",not (mouse_mean.groupby("mouse_id")[["age","age_group","sex"]].nunique()>1).any().any())
for node in main_axis["eligible"]:
    g=main_samples.loc[main_samples.node.eq(node)]
    fit=main_axis["fits"][node];scores=project_axis(g,fit,MAIN_N_PC)
    record_test("sign_and_origin_"+node,np.isclose(scores[g.age_group.eq("young")].mean(),0,atol=1e-5) and np.isclose(scores[g.age_group.eq("old")].mean(),fit["norm"],rtol=1e-5))
# Sex adjustment must remove the training-estimated female offset before projecting held cells.
sex_toy=pd.DataFrame({"mouse_id":["y1","y2","y3","o1","o2","o3","m1","m2"],"age_group":["young"]*3+["old"]*3+["middle"]*2,"sex":["male","female","male","male","female","male","male","female"],"PC1":[0,5,0,0,5,0,0,5],"PC2":[0,4,0,2,6,2,1,5]})
sex_fit=fit_axis(sex_toy,2,"sex_adjusted")
record_test("sex_adjusted_direction_recovers_age_effect",np.allclose(sex_fit["d"],[0,2]))
record_test("sex_standardised_scores_remove_sex_offset",np.allclose(project_axis(sex_toy,sex_fit,2),[0,0,0,2,2,2,1,1]))
test_table=pd.DataFrame(tests);assert test_table.passed.all()
test_table.to_csv("../results/aging_axis/current_cohort_20261030/simulation_and_data_checks.tsv",sep="\t",index=False)
params=dict(date="2026-10-30",n_pcs=MAIN_N_PC,min_cells=MAIN_MIN_CELLS,min_mice_per_endpoint=2,aggregation="mean",direction="old centroid - young centroid",age_groups="06 frozen; 24m=middle, 30m=old",cohort_mouse_nodes=117,cohort_mouse_ids=22,n_main_mouse_nodes=len(main_samples),n_primary_nodes=len(main_axis["eligible"]),primary_nodes=main_axis["eligible"],PCA_basis="06 cell-level X_pca; mean coordinates per mouse-node",validation="held mouse absent from direction/centre fitting; HVG/PCA basis frozen in 06, not fully nested",sex="unadjusted task-book centroid main; additive sex-adjusted and male-only controls; old all male",seed=SEED)
json.dump(params,open("../results/aging_axis/current_cohort_20261030/parameters.json","w",encoding="utf-8"),ensure_ascii=False,indent=1)
print(main_summary[["node","n_mice","n_young","n_old","status","rho_age_crossfit","min_lomo_cosine"]].to_string(index=False))


                    node  n_mice  n_young  n_old                      status  rho_age_crossfit  min_lomo_cosine
       Kidney | lymphoid      16        5      4                    complete          0.053778         0.862721
        Kidney | myeloid      16        5      4                    complete          0.479521         0.975547
        Liver | lymphoid       7        1      3 exploratory_low_replication          0.394665         0.962225
         Liver | myeloid       8        2      3                    complete          0.170783         0.691909
         Lung | lymphoid      16        6      4                    complete          0.057246         0.844842
          Lung | myeloid      16        6      4                    complete          0.259111         0.904951
spleen/marrow | lymphoid      19        5      4                    complete         -0.384185         0.097914
 spleen/marrow | myeloid      19        5      4                    complete          0.277219         0

In [4]:
# Continuous similarity and clustering; no significance stars or causal edges.
sim=main_axis["cosine"];D=np.clip(1-sim.to_numpy(),0,2);np.fill_diagonal(D,0)
Z=linkage(squareform(D),method="average")
order=leaves_list(Z);display_sim=sim.iloc[order,order]
fig=plt.figure(figsize=(10,9));tree_ax=fig.add_axes([.04,.29,.17,.61]);heat_ax=fig.add_axes([.23,.29,.64,.61])
dendrogram(Z,orientation="left",no_labels=True,ax=tree_ax,color_threshold=0,above_threshold_color="#777777");tree_ax.invert_yaxis();tree_ax.axis("off")
im=heat_ax.imshow(display_sim,vmin=-1,vmax=1,cmap="RdBu_r")
labels=[n.replace(" | ","\n") for n in display_sim.index]
heat_ax.set_xticks(range(len(labels)),labels,rotation=45,ha="right",fontsize=9)
heat_ax.set_yticks(range(len(labels)),labels,fontsize=9);heat_ax.yaxis.tick_right()
for i in range(len(labels)):
    for j in range(len(labels)):heat_ax.text(j,i,f"{display_sim.iloc[i,j]:.2f}",ha="center",va="center",fontsize=10)
cax=fig.add_axes([.23,.13,.55,.025]);fig.colorbar(im,cax=cax,orientation="horizontal",label="Cosine of old-young PCA directions")
fig.suptitle(f"Exploratory PCA direction similarity | PC1–{MAIN_N_PC} | current 08 cohort",fontsize=12)
fig.text(.04,.01,"Mouse-ID sampling unit; young/old >=2 IDs per displayed node. Descriptive similarity, not causality.\nLiver lymphoid excluded. Low-cell, sex and composite-ID sensitivities are reported in Week 11.\nNo node pair passed the joint operational robustness criteria; numeric similarity alone is not stable coupling evidence.",fontsize=8)
fig.savefig("../figures/main/Fig4_aging_similarity_draft.pdf",bbox_inches="tight")
fig.savefig("../figures/main/Fig4_aging_similarity_draft.png",dpi=170,bbox_inches="tight")
fig.savefig("../results/aging_axis/current_cohort_20261030/aging_similarity.pdf",bbox_inches="tight")
plt.close(fig)
fig,axes=plt.subplots(2,4,figsize=(16,7),layout="constrained")
for ax,node in zip(axes.flat,NODES):
    g=score_table.loc[score_table.node.eq(node)]
    for sex,marker in [("male","o"),("female","^")]:
        for age,color in [("young","#2166ac"),("middle","#999999"),("old","#b2182b")]:
            s=g.loc[g.sex.eq(sex)&g.age_group.eq(age)]
            ax.scatter(s.age,s.held_out_score,c=color,marker=marker,s=26)
    fit=main_axis["fits"][node]
    ax.set_title(node+"\n"+("primary" if fit["formal"] else "excluded / exploratory"),fontsize=9)
    ax.set_xlabel("Age (months)");ax.set_ylabel("Held-out axis score")
fig.suptitle("Leave-one-mouse-out projections | circles male, triangles female",fontsize=12)
fig.savefig("../results/aging_axis/current_cohort_20261030/held_out_age_scores.pdf")
fig.savefig("../results/aging_axis/current_cohort_20261030/held_out_age_scores.png",dpi=150)
plt.close(fig)
print("Saved Fig4 and held-out score figures. No P value is calculated from in-sample age separation.")


Saved Fig4 and held-out score figures. No P value is calculated from in-sample age separation.


## 2026-10-30交付补齐 · 当前冻结队列的Fig2交付
独立读取已有对象导出，不重跑上游拟合；运行目录为notebooks。历史版本保留在整理备份。


In [5]:
import json
import hashlib
import h5py
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy import sparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["font.sans-serif"]=["Microsoft YaHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
plt.rcParams["pdf.fonttype"]=42

with h5py.File("../data_processed/final_data.h5ad","r") as handle:
    fig2_obs=read_elem(handle["obs"]).copy();fig2_umap=np.asarray(read_elem(handle["obsm/X_umap"]))
fig2_mouse=pd.read_csv("../results/aging_axis/current_cohort_20261030/pca_scores.csv")
assert len(fig2_obs)==96569 and len(fig2_mouse)==117
fig2_mouse.to_csv("../figure_data/Fig2_mouse_PCA_and_heldout_scores.csv",index=False)
fig,axes=plt.subplots(2,2,figsize=(13,10),layout="constrained")
rng=np.random.default_rng(2026);idx=np.sort(rng.choice(len(fig2_obs),min(30000,len(fig2_obs)),replace=False))
for ax,field in zip(axes[0],["tissue","mct"]):
    for i,value in enumerate(sorted(fig2_obs[field].astype(str).unique())):
        hit=fig2_obs.iloc[idx][field].astype(str).eq(value).to_numpy()
        ax.scatter(fig2_umap[idx[hit],0],fig2_umap[idx[hit],1],s=1,alpha=.5,label=value,rasterized=True)
    ax.set_xlabel("Frozen UMAP1");ax.set_ylabel("Frozen UMAP2");ax.set_title("Current cell UMAP by "+field);ax.legend(markerscale=4,fontsize=8)
for group,color in [("young","#2166ac"),("middle","#999999"),("old","#b2182b")]:
    for mct,marker in [("lymphoid","o"),("myeloid","^")]:
        sub=fig2_mouse.loc[fig2_mouse.age_group.eq(group)&fig2_mouse.mct.eq(mct)]
        axes[1,0].scatter(sub.PC1,sub.PC2,s=25,c=color,marker=marker,alpha=.7,label=group+" / "+mct)
axes[1,0].set_xlabel("Mouse-node mean frozen PC1");axes[1,0].set_ylabel("PC2");axes[1,0].set_title("117 mouse-node samples; equal mouse-node weight");axes[1,0].legend(fontsize=7,ncol=2)
for node,g in fig2_mouse.groupby("node",observed=True):
    formal=node!="Liver | lymphoid"
    centres=g.groupby("age",observed=True).held_out_score.median()
    axes[1,1].plot(centres.index,centres.to_numpy(),marker="o",markersize=3,lw=.8,ls="-" if formal else "--",label=node)
axes[1,1].set_xlabel("Age (months)");axes[1,1].set_ylabel("Median held-out projection");axes[1,1].set_title("Descriptive held-out scores; exploratory node dashed");axes[1,1].legend(fontsize=6,ncol=2)
fig.suptitle("Frozen current cohort | 96569 cells,117 mouse-node samples\n24m=middle;30m=old. Fixed representation; no causal or robust shared-axis claim.",fontsize=12)
fig.savefig("../figures/main/Fig2_PCA_QC_draft.pdf");fig.savefig("../figures/main/Fig2_PCA_QC_draft.png",dpi=150);plt.close(fig)
print("Fig2 updated using frozen coordinates; cell drawing subsampled only for rendering.")


Fig2 updated using frozen coordinates; cell drawing subsampled only for rendering.
